In [1]:
import cv2
import numpy as np

In [2]:
img = cv2.imread('../img/img.jpg')
h,w = img.shape[:2]
left = img[:,:w//2]
right = img[:,w//2:]
cv2.imwrite('../img/a.jpg', left)
cv2.imwrite('../img/b.jpg', right)

True

In [3]:
img_a = cv2.imread('../img/a.jpg')
img_b = cv2.imread('../img/b.jpg')
gray_a = cv2.cvtColor(img_a,cv2.COLOR_BGR2GRAY)
gray_b = cv2.cvtColor(img_b,cv2.COLOR_BGR2GRAY)

In [4]:
sift = cv2.SIFT_create()
kp_a, des_a = sift.detectAndCompute(gray_a,None)
kp_b, des_b = sift.detectAndCompute(gray_b,None)

In [5]:
index_params = dict(algorithm=1,trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params,search_params)
matches = flann.knnMatch(des_a,des_b,k=2)

In [6]:
good_matches = []
for m,n in matches:
    if m.distance < 0.7*n.distance:
        good_matches.append(m)
        
matched_keypoints_img = cv2.drawMatches(
    img_a,kp_a,img_b,kp_b,good_matches,
    None,flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)
src_pts = np.float32([kp_b[m.trainIdx].pt for m in good_matches]).reshape(-1,1,2)
dst_pts = np.float32([kp_a[m.queryIdx].pt for m in good_matches]).reshape(-1,1,2)

H,_ = cv2.findHomography(src_pts,dst_pts,cv2.RANSAC,5.0)

In [7]:
h_a,w_a = img_a.shape[:2]
h_b,w_b = img_b.shape[:2]

pts = np.float32([[0,0],[0,h_b],[w_b,h_b]]).reshape(-1,1,2)
dst_corners = cv2.perspectiveTransform(pts,H)
all_corners = np.concatenate([dst_corners,np.float32([[0,0],[w_a,0],[w_a,h_a],[0,h_a]]).reshape(-1,1,2)],axis=0)
[x_min,y_min] = np.int32(all_corners.min(axis=0).ravel()-0.5)
[x_max,y_max] = np.int32(all_corners.max(axis=0).ravel()+0.5)
translation_matrix = np.array([[1,0,-x_min],[0,1,-y_min],[0,0,1]],dtype=np.float32)

In [8]:
stitched_img = cv2.warpPerspective(
    img_b,
    translation_matrix @ H,
    (x_max-x_min,y_max-y_min)
)
stitched_img[-y_min:h_a-y_min,-x_min:w_a-x_min] = img_a

In [10]:
cv2.imwrite('../img/matched_keypoints_img.jpg', matched_keypoints_img)
cv2.imwrite('../img/stitched_img.jpg', stitched_img)
cv2.imshow('matched keypoint', matched_keypoints_img)
cv2.imshow('result', stitched_img)
cv2.waitKey(0)
cv2.destroyAllWindows()